Imports

In [4]:
import os
import glob
import numpy as np
import rasterio
from rasterio.plot import show
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    f1_score,
    classification_report
)
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt
import seaborn as sns

processed_dir = "../data/processed/"
held_out_fire = "thomas_fire"

feature_names = ['ndvi', 'nbr', 'esa_landcover', 'era5_mean_temp', 'chirps_precip', 'srtm_dem', 'slope', 'aspect', 'burned_arr']

Load raster as array helper function

In [5]:
def load_raster(path):
    with rasterio.open(path) as src:
        arr = src.read().astype("float32")
        profile = src.profile
    return arr, profile

Load training fires

In [6]:
train_features_list = []
train_labels_list = []

for fire_folder in os.listdir(processed_dir):
    if fire_folder == held_out_fire:
        continue
    
    feat_path = os.path.join(processed_dir, fire_folder, "features.tif")
    label_path = os.path.join(processed_dir, fire_folder, "labels.tif")

    if not os.path.exists(feat_path) or not os.path.exists(label_path):
        print(f"Skipping: {fire_folder} (missing files)")
        continue
    
    X, _ = load_raster(feat_path)
    y, _ = load_raster(label_path)

    X = X.reshape(X.shape[0], -1).T  
    y = y.reshape(-1)

    mask = (y == 0) | (y == 1)
    X = X[mask]
    y = y[mask]

    train_features_list.append(X)
    train_labels_list.append(y)

# Combine into single training set
X_train = np.vstack(train_features_list)
y_train = np.hstack(train_labels_list)

print("Training data loaded:")
print("  X_train:", X_train.shape)
print("  y_train:", y_train.shape)

Training data loaded:
  X_train: (110236, 9)
  y_train: (110236,)


Load held out fire test set

In [7]:
feat_path = os.path.join(processed_dir, held_out_fire, "features.tif")
label_path = os.path.join(processed_dir, held_out_fire, "labels.tif")

X_test, test_profile = load_raster(feat_path)
y_test, _ = load_raster(label_path)

X_test = X_test.reshape(X_test.shape[0], -1).T
y_test = y_test.reshape(-1)

mask = (y_test == 0) | (y_test == 1)
X_test = X_test[mask]
y_test = y_test[mask]

print("Held-out Thomas fire loaded:")
print("  X_test:", X_test.shape)
print("  y_test:", y_test.shape)

Held-out Thomas fire loaded:
  X_test: (24120, 9)
  y_test: (24120,)


Scale features for Logistic Regression

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)